# Cifra de César

A Cifra de César é uma cifra de substituição monoalfabética em que cada letra do texto original (texto claro) é substituída pela letra que está **k** posições à frente no alfabeto. O valor **k** é a chave (deslocamento).

Fórmula de encriptação para uma letra de índice `x` (A = 0, B = 1, ..., Z = 25):

$$E_k(x) = (x + k) \bmod 26$$

Neste exercício a chave **k** pode variar de **1 a N**. A primeira parte faz a **encriptação**. Depois, uma criptoanálise por frequência de letras tenta recuperar a chave a partir do texto cifrado.

## Implementação

Regras adotadas:

- Letras maiúsculas e minúsculas são deslocadas mantendo a caixa.
- Caracteres que não são letras (espaços, números, pontuação) são mantidos sem alteração.
- A chave é reduzida com `k % 26`, então qualquer `k >= 1` é aceito (k = 27 equivale a k = 1, por exemplo).

In [21]:
TAMANHO_ALFABETO = 26


def encriptar(texto: str, k: int) -> str:
    """Encripta `texto` com a Cifra de César usando deslocamento `k` (k >= 1)."""
    if not isinstance(k, int) or k < 1:
        raise ValueError("A chave k deve ser um inteiro maior ou igual a 1.")

    k = k % TAMANHO_ALFABETO
    resultado = []

    for caractere in texto:
        if "A" <= caractere <= "Z":
            base = ord("A")
            resultado.append(chr((ord(caractere) - base + k) % TAMANHO_ALFABETO + base))
        elif "a" <= caractere <= "z":
            base = ord("a")
            resultado.append(chr((ord(caractere) - base + k) % TAMANHO_ALFABETO + base))
        else:
            resultado.append(caractere)

    return "".join(resultado)

## Exemplos

In [22]:
print(encriptar("ATAQUE AO AMANHECER", 3))   # DWDTXH DR DPDQKHFHU
print(encriptar("Cifra de Cesar", 1))         # Djgsb ef Dftbs
print(encriptar("xyz XYZ", 3))                # abc ABC  (volta ao início do alfabeto)
print(encriptar("Python", 13))          # Clguba 3.11!  (números e pontuação não mudam)
print(encriptar("abc", 27))                   # bcd  (k = 27 equivale a k = 1)

DWDTXH DR DPDQKHFHU
Djgsb ef Dftbs
abc ABC
Clguba
bcd


## Encriptando com todas as chaves de 1 a N

Como a chave pode variar de 1 a N, a célula abaixo mostra o resultado da encriptação de uma mesma mensagem para cada valor de k nesse intervalo.

In [23]:
def encriptar_intervalo(texto: str, n: int) -> dict[int, str]:
    """Retorna um dicionário {k: texto_encriptado} para k de 1 a n."""
    if n < 1:
        raise ValueError("N deve ser maior ou igual a 1.")
    return {k: encriptar(texto, k) for k in range(1, n + 1)}


mensagem = "CIFRA DE CESAR"
N = 25

for k, cifrado in encriptar_intervalo(mensagem, N).items():
    print(f"k = {k:2d}: {cifrado}")

k =  1: DJGSB EF DFTBS
k =  2: EKHTC FG EGUCT
k =  3: FLIUD GH FHVDU
k =  4: GMJVE HI GIWEV
k =  5: HNKWF IJ HJXFW
k =  6: IOLXG JK IKYGX
k =  7: JPMYH KL JLZHY
k =  8: KQNZI LM KMAIZ
k =  9: LROAJ MN LNBJA
k = 10: MSPBK NO MOCKB
k = 11: NTQCL OP NPDLC
k = 12: OURDM PQ OQEMD
k = 13: PVSEN QR PRFNE
k = 14: QWTFO RS QSGOF
k = 15: RXUGP ST RTHPG
k = 16: SYVHQ TU SUIQH
k = 17: TZWIR UV TVJRI
k = 18: UAXJS VW UWKSJ
k = 19: VBYKT WX VXLTK
k = 20: WCZLU XY WYMUL
k = 21: XDAMV YZ XZNVM
k = 22: YEBNW ZA YAOWN
k = 23: ZFCOX AB ZBPXO
k = 24: AGDPY BC ACQYP
k = 25: BHEQZ CD BDRZQ


## Criptoanálise por ordem de ocorrência

Conta-se a frequência de cada letra no texto cifrado e ordena-se da mais comum para a menos comum. Faz-se o mesmo com a tabela do português e associa-se posição a posição: a letra mais comum do cifrado vira **A**, a segunda vira **E**, a terceira vira **O**, e assim por diante. As letras mais frequentes tendem a acertar; as de frequência parecida saem trocadas.

In [24]:
# Frequências relativas do português, em porcentagem.
# Fonte: https://pt.wikipedia.org/wiki/Frequ%C3%AAncia_de_letras
FREQUENCIA_PORTUGUES = {
    "a": 14.63,
    "b": 1.04,
    "c": 3.88,
    "d": 4.99,
    "e": 12.57,
    "f": 1.02,
    "g": 1.30,
    "h": 1.28,
    "i": 6.18,
    "j": 0.40,
    "k": 0.02,
    "l": 2.78,
    "m": 4.74,
    "n": 5.05,
    "o": 10.73,
    "p": 2.52,
    "q": 1.20,
    "r": 6.53,
    "s": 7.81,
    "t": 4.34,
    "u": 4.63,
    "v": 1.67,
    "w": 0.01,
    "x": 0.21,
    "y": 0.01,
    "z": 0.47,
}



In [25]:
TEXTO = """
Em 1936, Alan Turing publicou um artigo que mudou a maneira de pensar o cálculo. Em vez de descrever uma máquina para cada problema, ele imaginou um dispositivo simples, com uma fita infinita e um conjunto pequeno de regras. Essa máquina universal podia imitar qualquer procedimento mecânico, desde que as instruções estivessem escritas na própria fita. Anos depois, essa ideia ganhou corpo nos computadores de programa armazenado: os dados e as instruções passaram a ocupar a mesma memória.

Durante a Segunda Guerra Mundial, Turing trabalhou em Bletchley Park na leitura de mensagens cifradas. O desafio não era apenas experimentar deslocamentos, como na cifra de César. As máquinas usadas pelo Eixo trocavam a substituição a cada letra, e uma busca manual era impossível. A resposta foi mecanizar a criptoanálise. Comparar o texto cifrado com padrões conhecidos da língua, e descartar milhares de chaves absurdas, virou um trabalho de máquina. O computador moderno nasce, em parte, desse encontro entre lógica e frequência de letras.

Há uma lição direta para a cifra de César. Como cada letra anda o mesmo número de casas, o texto cifrado conserva o perfil da língua, apenas deslocado. Se o português usa muito a letra A, o texto cifrado usa muito a letra que está k posições adiante. Medir a distância entre a frequência observada e a frequência esperada da língua revela a chave, sem que ela tenha sido informada. Quanto mais longo e mais comum for o texto, mais confiável fica essa medida. Uma página de prosa já costuma bastar. Uma frase curta, cheia de nomes próprios, pode enganar.

Esse relato acompanha o nascimento do computador, da máquina de Turing aos equipamentos que testavam chaves em escala. A mesma ideia segue presente: um programa é uma sequência de instruções guardadas na memória, capaz de tratar texto, números e outras instruções como dados. Foi esse salto que tornou a criptoanálise, e depois toda a computação, uma tarefa automática.
""".strip()

In [26]:
import unicodedata
from collections import Counter


def normalizar(texto: str) -> str:
    """Remove acentos (á -> a, ç -> c) para que todas as letras entrem na contagem."""
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")


def frequencias(texto: str) -> dict[str, float]:
    """Porcentagem de ocorrência de cada letra (a-z) no texto."""
    letras = [c.lower() for c in texto if "a" <= c.lower() <= "z"]
    contagem = Counter(letras)
    total = len(letras)
    return {letra: 100 * contagem.get(letra, 0) / total for letra in FREQUENCIA_PORTUGUES}


def tabela_por_frequencia(cifrado: str) -> dict[str, str]:
    """Associa as letras do cifrado às do português pela posição no ranking de frequência.

    A letra mais comum do cifrado vira A, a segunda vira E, a terceira vira O, e assim por diante.
    """
    observadas = frequencias(cifrado)
    ordem_cifrado = sorted(observadas, key=lambda letra: -observadas[letra])
    ordem_portugues = sorted(FREQUENCIA_PORTUGUES, key=lambda letra: -FREQUENCIA_PORTUGUES[letra])
    return dict(zip(ordem_cifrado, ordem_portugues))


def substituir(texto: str, tabela: dict[str, str]) -> str:
    """Troca cada letra pela indicada na tabela, mantendo a caixa e os demais caracteres."""
    resultado = []
    for caractere in texto:
        minuscula = caractere.lower()
        if "a" <= minuscula <= "z":
            trocada = tabela[minuscula]
            resultado.append(trocada.upper() if caractere.isupper() else trocada)
        else:
            resultado.append(caractere)
    return "".join(resultado)


def trecho(texto: str, palavras: int = 20) -> str:
    """Devolve as primeiras `palavras` palavras do texto, numa linha só."""
    return " ".join(texto.split()[:palavras])


# 1. Cifra o texto sobre Turing com uma chave qualquer.
CHAVE = 11
texto_claro = normalizar(TEXTO)
cifrado = encriptar(texto_claro, CHAVE)

print(f"Chave usada na cifragem: k = {CHAVE}")
print()

# 2. Conta as letras do cifrado e monta a tabela pela ordem de frequência.
tabela = tabela_por_frequencia(cifrado)
observadas = frequencias(cifrado)

print("cifrada  observada  ->  suposta  esperada   acertou?")
for letra_cifrada, letra_suposta in tabela.items():
    letra_real = encriptar(letra_cifrada, TAMANHO_ALFABETO - CHAVE)  # desfaz o deslocamento
    acertou = letra_real == letra_suposta
    print(
        f"   {letra_cifrada}      {observadas[letra_cifrada]:5.2f}%   ->     {letra_suposta}     "
        f"{FREQUENCIA_PORTUGUES[letra_suposta]:5.2f}%     {'sim' if acertou else 'nao'}"
    )

# 3. Converte o cifrado com a tabela e compara com o texto original.
tentativa = substituir(cifrado, tabela)
total_letras = sum("a" <= c.lower() <= "z" for c in texto_claro)
letras_certas = sum(a == b for a, b in zip(tentativa, texto_claro) if "a" <= b.lower() <= "z")

print()
print(f"Letras no lugar certo: {letras_certas}/{total_letras} ({100 * letras_certas / total_letras:.1f}%)")
print()

# 4. Mostra as 20 primeiras palavras de cada versão para comparar.
print("Original:  ", trecho(texto_claro))
print("Cifrado:   ", trecho(cifrado))
print("Convertido:", trecho(tentativa))

Chave usada na cifragem: k = 11
Texto cifrado:
Px 1936, Lwly Efctyr afmwtnzf fx lcetrz bfp xfozf l xlyptcl op apydlc z nlwnfwz. Px gpk op opdncpgpc fxl xlbftyl alcl nlol aczmwpxl, pwp txlrtyzf fx otdazdtetgz dtxawpd, nzx fxl qtel tyqtytel p fx nzyufyez apbfpyz op cprcld. Pddl xlbftyl fytgpcdlw azotl txtelc bflwbfpc acznpotxpyez xpnlytnz, opdop bfp ld tydecfnzpd pdetgpddpx pdncteld yl aczactl qtel. Lyzd opaztd, pddl toptl rlyszf nzcaz yzd nzxafelozcpd op aczrclxl lcxlkpyloz: zd olozd p ld tydecfnzpd alddlclx l znfalc l xpdxl xpxzctl.

Ofclyep l Dprfyol Rfpccl Xfyotlw, Efctyr eclmlwszf px Mwpenswpj Alcv yl wptefcl op xpydlrpyd ntqclold. Z opdlqtz ylz pcl lapyld piapctxpyelc opdwznlxpyezd, nzxz yl ntqcl op Npdlc. Ld xlbftyld fdlold apwz Ptiz ecznlglx l dfmdeteftnlz l nlol wpecl, p fxl mfdnl xlyflw pcl txazddtgpw. L cpdazdel qzt xpnlytklc l nctaezlylwtdp. Nzxalclc z epiez ntqcloz nzx aloczpd nzyspntozd ol wtyrfl, p opdnlcelc xtwslcpd op nslgpd lmdfcold, gtczf fx eclmlwsz op xlbftyl. Z nzxa